In [ ]:
# Install dependencies if missing
# !pip install SpeechRecognition pyaudio pydub


## Speech Recognition in Python using Google Speech API

In [3]:
import os
from pydub import AudioSegment

# Look for audio files
audio_files = [f for f in os.listdir('.') if f.endswith(('.wav', '.mp3', '.m4a', '.flac'))]

filename = None
target_wav = "processed_audio.wav"

if audio_files:
    # Prioritize non-processed files
    candidates = [f for f in audio_files if f != target_wav]
    filename = candidates[0] if candidates else audio_files[0]
    print(f"Found audio file: {filename}")
    
    # If it's not a WAV, convert it using pydub
    if not filename.endswith('.wav'):
        print(f"Converting {filename} to WAV format...")
        try:
            audio = AudioSegment.from_file(filename)
            # Limit to first 60 seconds if it's a long song (Google Free API limit)
            if len(audio) > 60000:
                print("Audio is long. Trimming to first 60 seconds for processing...")
                audio = audio[:60000]
            
            audio.export(target_wav, format="wav")
            filename = target_wav
            print("Conversion and Trimming successful!")
        except Exception as e:
            print(f"Conversion failed: {e}.")
            print("TIP: Ensure 'ffmpeg' is installed and in your PATH for MP3 support.")
            filename = None
else:
    print("No audio file found. Falling back to microphone.")

No audio file found. Please provide a .wav file or use the microphone logic below.


In [ ]:
import speech_recognition as sr
recognizer = sr.Recognizer()

if filename and os.path.exists(filename):
    try:
        with sr.AudioFile(filename) as source:
            print(f"Analyzing audio: {filename}...")
            # Adjust for noise
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio_data = recognizer.record(source)
        
        print("\n--- Transcription Result ---")
        text = recognizer.recognize_google(audio_data)
        print(text)
    except sr.UnknownValueError:
        print("Google Speech Recognition could not understand the audio.")
    except sr.RequestError as e:
        print(f"Could not request results from Google service; {e}")
    except Exception as e:
        print(f"An error occurred during processing: {e}")
else:
    print("\n--- Starting Microphone Capture ---")
    print("Speak now (listening for 5 seconds)...")
    try:
        with sr.Microphone() as source:
            recognizer.adjust_for_ambient_noise(source, duration=1)
            audio_data = recognizer.listen(source, timeout=10, phrase_time_limit=10)
            
        print("Transcribing...")
        text = recognizer.recognize_google(audio_data)
        print(f"\nRecognized Text: {text}")
    except sr.WaitTimeoutError:
        print("Listening timed out. No speech detected.")
    except Exception as e:
        print(f"An error occurred with microphone: {e}")